# How many bikes each plant assembles each month, with stock at the plants

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/plant_stock.ipynb)

A company assembles electric bicycles in three plants and plans, from January to June, how many bikes each plant assembles each month. In the first version of the model, M0, the demand of a month is assembled in that month, and with a second demand forecast the model has no solution, although the plants could serve that demand by assembling bikes in advance. This notebook uses the second version, M1, which keeps bikes in stock at each plant from one month to the next. It follows the plan in steps:

1. The data of the plants and the demand forecast.
2. The model with stock at the plants.
3. The plan with forecast 2, with and without a stock required at the end of June.
4. Four tests, each one changing a single datum: the holding cost, the demand of one month, the last month of the horizon and the stock required at the end.

**How to use it.** Run the cells in order, with ▶ or *Shift + Enter*. Some cells have a form (sliders, drop-down lists and tick boxes): change the values and run the cell again. Outside Colab the form is a line such as `last_month = "Mar"`: edit the value and run the cell.

**Symbols.** The column «From» gives the version of the model where each symbol first appears.

| Symbol | Meaning | Unit | From |
|---|---|---|---|
| $F$, $f$ | Plants | — | M0 |
| $T$, $t$ | Months, January to June; $t = 0$ is the end of December | — | M0 |
| $\tau$ | Last month of the horizon | — | M1 |
| $c_f$ | Unit assembly cost at plant $f$ | €/bike | M0 |
| $K_{ft}$ | Assembly capacity of plant $f$ in month $t$ | bikes/month | M0 |
| $d_t$ | Orders received in month $t$ | bikes/month | M0 |
| $c^h_f$ | Holding cost at plant $f$ | €/(bike·month) | M1 |
| $h_{f0}$ | Stock at plant $f$ at the end of December | bikes | M1 |
| $h^{\text{end}}_f$ | Stock that plant $f$ must hold at the end of the horizon | bikes | M1 |
| $p_{ft}$ | Bikes assembled at plant $f$ in month $t$ | bikes/month | M0 |
| $h_{ft}$ | Bikes in stock at plant $f$ at the end of month $t$ | bikes | M1 |
| $x_{ft}$ | Bikes shipped from plant $f$ in month $t$ | bikes/month | M1 |

All the data and results are explained in the [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/plant_stock.md). The data are invented. Running the notebook is optional: it is support material, not part of the assessment.

In [ ]:
# @title Setup (run this cell first) { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import plant_stock as ps

## 1. Plants and demand

The first table gives, for each plant, the cost of assembling one bike (€/bike) and its capacity (bikes/month, the same every month), as in M0, and the three data that M1 adds: the cost of holding one bike in stock for one month, the stock at the end of December and the stock that the plan must leave at the end of the horizon, equal to the stock at the start. The second table is forecast 2 of M0, the one with which M0 has no solution.

In [ ]:
# @title Data { display-mode: "form" }
ps.show_data()

## 2. The model with stock at the plants

$$
\begin{aligned}
\min \quad & \sum_{f \in F} \sum_{t \in T} \left( c_f\, p_{ft} + c^h_f\, h_{ft} \right) && && \text{(assembly and holding cost)} \\
\text{s.t.} \quad & h_{f,t-1} + p_{ft} = x_{ft} + h_{ft} && \forall f \in F,\ t \in T && \text{(stock balance at each plant)} \\
& x_{ft} \le h_{f,t-1} + p_{ft} && \forall f \in F,\ t \in T && \text{(shipment limit at each plant)} \\
& \sum_{f \in F} x_{ft} \ge d_t && \forall t \in T && \text{(demand of each month)} \\
& p_{ft} \le K_{ft} && \forall f \in F,\ t \in T && \text{(capacity)} \\
& h_{f\tau} \ge h^{\text{end}}_f && \forall f \in F && \text{(final stock; only in the runs with a target)} \\
& p_{ft},\ x_{ft},\ h_{ft} \ge 0 && \forall f \in F,\ t \in T
\end{aligned}
$$

with $h_{f0}$ given. What a plant holds at the start of a month plus what it assembles is what it ships plus what it holds at the end of the month. A plant cannot ship more than it held at the start of the month plus what it assembles that month. The plants together ship at least the demand of each month. In M0 what a plant assembles in a month is what it delivers that month; with stock the two are different, and the model has one variable for each.

## 3. Plan with forecast 2

The cell solves the model with forecast 2 and gives, for each plant and month, the bikes assembled, the bikes shipped and the bikes in stock at the end of the month, with the cost of the plan. Then the **shadow price** of the demand of each month (€/bike): what one more bike of demand that month would cost. The figure on the left stacks the bikes assembled by each plant against the capacity of the three plants and the demand; the one on the right stacks the stock of each plant at the end of each month, starting from December.

The tick box requires, or not, the stock at the end of June given in the data. Run the cell with and without it. In which months do the two plans differ, and why does the plan without the requirement end June with no stock? Which plant holds the stock built before April, and why that one?

In [ ]:
# @title Plan with forecast 2 { display-mode: "form" }
final_stock_target = True  # @param {type:"boolean"}
ps.stock_plan(final_stock_target);

## 4. The holding cost

The slider sets the holding cost of plant 1, in €/(bike·month). The holding costs of plants 2 and 3 move with it, 1 and 2 €/(bike·month) higher, so that holding stock is still cheapest at plant 1 and the plan has one answer. Holding a bike is what assembling it in advance costs; the other way to serve April and May is plant 3, which costs 50 €/bike more than plant 2. The cell gives the bikes in stock at the end of March, the bikes assembled at plant 3, and then the plan as in the previous cell.

Move the holding cost up from 5. At what holding cost does plant 3 start to assemble in April, and which bikes assembled in advance does it replace? How early does the plan assemble bikes when holding them is expensive?

In [ ]:
# @title Holding cost { display-mode: "form" }
holding_cost_plant_1 = 5  # @param {type:"slider", min:1, max:40, step:1}
ps.holding_cost_test(holding_cost_plant_1);

## 5. The demand of one month

Choose a month and its demand; the other months keep forecast 2, and the plan keeps the stock required at the end of June. While the demand since January does not exceed what the plants can assemble since January plus the stock they held at the start, the plan assembles bikes in advance to serve it. When it does, the model has no solution, and the cell says in which month capacity falls short and by how many bikes.

Raise the demand of April step by step. Up to what demand does the model find a plan, and in which months are the extra bikes assembled? When there is no plan, does the company have no way to serve that demand, or does the model leave out something it could do?

In [ ]:
# @title Demand of one month { display-mode: "form" }
month = "Apr"  # @param ["Jan", "Feb", "Mar", "Apr", "May", "Jun"]
demand = 650  # @param {type:"slider", min:0, max:1500, step:10}
ps.demand_test(month, demand);

## 6. The last month of the horizon

The cell makes a plan from January to the month you choose, which must leave the plants at the end of that month with the stock required in the data. Then it makes a second plan for the months left until June, starting from the stock that the first plan leaves. With June there is only one plan, the one of section 3.

Choose March. Does the plan from January to March build stock for April? What happens to the plan made in April? Try April and May: how long does the horizon have to be for the first plan to see the peak of demand?

In [ ]:
# @title Last month of the horizon { display-mode: "form" }
last_month = "Mar"  # @param ["Mar", "Apr", "May", "Jun"]
ps.horizon_test(last_month);

## 7. The stock required at the end of June

The stock at the end of June is not decided by this plan: it is a requirement that comes from outside, for example from the plan for the next months. Choose a plant and the stock it must hold at the end of June; the other plants keep the stock of the data. The cell gives the plan with forecast 2 and that requirement, or says why there is no plan.

Raise the stock required at plant 1. In which months does the plan assemble the extra bikes, and at which plants? How much does the cost of the plan grow for each extra bike required?

In [ ]:
# @title Stock required at the end of June { display-mode: "form" }
plant = "Plant 1"  # @param ["Plant 1", "Plant 2", "Plant 3"]
stock_at_the_end = 20  # @param {type:"slider", min:0, max:300, step:10}
ps.final_stock_test(plant, stock_at_the_end);

All the numbers are explained in the [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/plant_stock.md), which also has a script that reproduces them.